In [ ]:
# PySpark Setup - Run this cell first
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pyspark -q
    !pip install findspark -q
    import findspark
    findspark.init()


# REFERENCE
https://github.com/GoogleCloudDataproc/spark-bigquery-connector

### VERTEXAI and DATAPROC

https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/workbench/spark/spark_bigquery.ipynb

## Method 1: Using BigQuery Client

In [1]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/prashantsingh/Learning/ipynb/service_account_key.json"

In [2]:
from google.oauth2 import service_account
from google.cloud import bigquery 
from pyspark.sql import SparkSession,Row

In [ ]:
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('Connection with Hive') \
        .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
        .config("hive.metastore.uris", "thrift://localhost:9083") \
        .enableHiveSupport() \
        .getOrCreate()
sc = spark.sparkContext

In [ ]:
def executeSql(query):
    try:
        client = bigquery.Client()
    except Exception as ex:
        print(str(ex))
        
    job_config = bigquery.QueryJobConfig()
    query_job = client.query(query,job_config=job_config)
    rows = query_job.result()
    df = None
    try:
        rows_data = [dict(row.items())for row in rows]
        df = spark.createDataFrame(Row(**row) for row in rows_data)
    except Exception as ex:
        print(str(ex))
    return df
    

In [ ]:
sql = f''' SELECT * FROM long-disk-418007.naveen_sample_dataset.student '''
df = executeSql(sql)

In [ ]:
df.show()

## Method 2.1: Using spark.read.format() API

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('Connection with Hive') \
        .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
        .config("hive.metastore.uris", "thrift://localhost:9083") \
        .config('spark.jars', './spark-3.5-bigquery-0.37.0.jar') \
        .enableHiveSupport() \
        .getOrCreate()
sc = spark.sparkContext

In [ ]:
project_id = 'long-disk-418007'
project_dataset = 'naveen_sample_dataset'
bq_table = 'student'
service_account_key = './service_account_key.json'

In [ ]:
df = spark.read.format('bigquery') \
        .option("credentialsFile",service_account_key) \
        .option("parentProject",project_id) \
        .option("project",project_id) \
        .option("dataset",project_dataset) \
        .load(bq_table)

In [ ]:
df.show()

## Method 2.2: Using spark.read.format() API

In [3]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/prashantsingh/Learning/ipynb/service_account_key.json"

In [4]:
from pyspark.sql import SparkSession

In [5]:
project_dataset = 'naveen_sample_dataset'

In [6]:
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('Connection with Hive') \
        .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
        .config("hive.metastore.uris", "thrift://localhost:9083") \
        .config('spark.jars', './spark-3.5-bigquery-0.37.0.jar') \
        .config('viewsEnabled', 'true') \
        .config('materializationDataset', project_dataset) \
        .enableHiveSupport() \
        .getOrCreate()
sc = spark.sparkContext
sql = f''' SELECT * FROM long-disk-418007.naveen_sample_dataset.student '''
df = spark.read.format("bigquery").load(sql)
df.show()

Exception in thread "main" java.lang.NoClassDefFoundError: org/apache/hadoop/fs/FSDataInputStream
	at java.lang.Class.getDeclaredMethods0(Native Method)
	at java.lang.Class.privateGetDeclaredMethods(Class.java:2701)
	at java.lang.Class.privateGetMethodRecursive(Class.java:3048)
	at java.lang.Class.getMethod0(Class.java:3018)
	at java.lang.Class.getMethod(Class.java:1784)
	at sun.launcher.LauncherHelper.validateMainClass(LauncherHelper.java:669)
	at sun.launcher.LauncherHelper.checkAndLoadMain(LauncherHelper.java:651)
Caused by: java.lang.ClassNotFoundException: org.apache.hadoop.fs.FSDataInputStream
	at java.net.URLClassLoader.findClass(URLClassLoader.java:387)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:418)
	at sun.misc.Launcher$AppClassLoader.loadClass(Launcher.java:355)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:351)
	... 7 more


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
sql = f''' SELECT * FROM long-disk-418007.naveen_sample_dataset.student '''
df = spark.read.format("bigquery").load(sql)
df.show()

In [ ]:
df.write \
  .format("bigquery") \
  .option("writeMethod", "direct") \
  .save(f"{project_dataset}.student1")

## Method 2.3: Using spark.read.format() API and BigQuery Client

In [ ]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/prashantsingh/Learning/ipynb/service_account_key.json"

In [ ]:
from pyspark.sql import SparkSession
from google.cloud import bigquery 

In [ ]:
project_dataset = 'naveen_sample_dataset'

In [ ]:
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('Connection with Hive') \
        .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
        .config("hive.metastore.uris", "thrift://localhost:9083") \
        .config('spark.jars', './spark-3.5-bigquery-0.37.0.jar') \
        .config('viewsEnabled', 'true') \
        .config('materializationDataset', project_dataset) \
        .enableHiveSupport() \
        .getOrCreate()
sc = spark.sparkContext

In [ ]:
def executeSQL(sql,spark = None, DDL = False):
    df = None
    if DDL==False:
        df = spark.read.format("bigquery").load(sql)
    else:
        try:
            client = bigquery.Client()
        except Exception as ex:
            print(str(ex))
            
        job_config = bigquery.QueryJobConfig()
        query_job = client.query(sql,job_config=job_config)
        rows = query_job.result()
        print('Query executed successfully')
    return df
        

In [ ]:
sql = f''' DROP TABLE long-disk-418007.naveen_sample_dataset.student2  '''

In [ ]:
df = executeSQL(sql,spark = spark,DDL = True)

In [ ]:
df.show()